# 01 - Exploratory Data Analysis (EDA)
## PRISM: Predictive Recurrent Infiltration State Model

This notebook explores the raw network traffic telemetry from the CSE-CIC-IDS2018 dataset, examining class distributions, protocol breakdowns, port frequencies, and feature skewness.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure styling
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
%matplotlib inline

### 1. Load Processed Traffic Sample

In [ ]:
data_path = Path('data/splits/train.npz')
if data_path.exists():
    train_data = np.load(data_path)
    states = train_data['states']
    labels_binary = train_data['labels_binary']
    labels_mitre = train_data['labels_mitre']
    print(f'Train states shape: {states.shape}')
    print(f'Attack windows: {labels_binary.sum()} / {len(labels_binary)} ({100*labels_binary.mean():.1f}%)')
else:
    print('Train splits not found. Please run feature extraction or split_dataset.py.')

### 2. MITRE Stage Distribution in Real Telemetry

In [ ]:
from src.utils.constants import MITRE_STAGES_INV, MITRE_STAGE_COLORS

stages, counts = np.unique(labels_mitre, return_counts=True)
stage_names = [MITRE_STAGES_INV.get(s, f'Stage {s}') for s in stages]

plt.figure(figsize=(10, 5))
bars = plt.bar(stage_names, counts, color=[MITRE_STAGE_COLORS.get(name, '#3498db') for name in stage_names])
plt.title('Distribution of Network State Windows by MITRE ATT&CK Stage', fontsize=14, fontweight='bold')
plt.xlabel('MITRE Stage')
plt.ylabel('Window Count (W=30s)')
plt.xticks(rotation=25)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 10, int(yval), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

### 3. Feature Correlations & State Variance Analysis

In [ ]:
sub_features = states[:, :10]
corr = np.corrcoef(sub_features, rowvar=False)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True)
plt.title('Correlation Matrix Across Top 10 State Features', fontsize=12)
plt.tight_layout()
plt.show()